# CISO Training & Benchmarking Pipeline
Train CISO and benchmark it under the STEM-LM evaluation protocol.

**Steps:**
1. Setup
2. Load and inspect data
3. Prepare files in CISO's expected format
4. Verify integrity
5. Write config
6. Train
7. Random-mask test sweep at p in {0.25, 0.5, 0.75, 1.0}
8. Recompute STEM-LM metric set (masked-only per-species filter)


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

# ── Paths ─────────────────────────────────────────────────────────────────────
CISO_DIR  = '/content/CISO-SDM'
DATA_DIR  = '/content/drive/MyDrive/CISO/data'
CKPT_DIR  = '/content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339'
RESULTS_DIR = '/content/drive/MyDrive/CISO/results_CISO_plant_50_epochs_1339'

# ── Clone repo locally ────────────────────────────────────────────────────────
if not os.path.exists(CISO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/RolnickLab/CISO-SDM', CISO_DIR], check=True)
else:
    print("Repo already cloned, skipping.")

os.chdir(CISO_DIR)
print("Working directory:", os.getcwd())

# ── Create Drive folders ──────────────────────────────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
# ── Symlink data/ → Drive ─────────────────────────────────────────────────────
if not os.path.exists(f'{CISO_DIR}/data'):
    os.symlink(DATA_DIR, f'{CISO_DIR}/data')
    print("Symlinked data/ → Drive")
else:
    print("Symlink already exists, skipping.")

# ── Install dependencies locally ─────────────────────────────────────────────
!pip install "numpy<2.0" -q
!pip install -r requirements.txt -q

print("\nSetup complete ✓")
print(f"  repo:         {CISO_DIR}")
print(f"  data (Drive): {DATA_DIR}")
print(f"  ckpts (Drive): {CKPT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already cloned, skipping.
Working directory: /content/CISO-SDM
Symlink already exists, skipping.

Setup complete ✓
  repo:         /content/CISO-SDM
  data (Drive): /content/drive/MyDrive/CISO/data
  ckpts (Drive): /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339


In [ ]:
import numpy as np
import pandas as pd
import json
import yaml
import shutil
from pathlib import Path

print("Imports OK")

Imports OK


Upload modified main.py with CSVLogger, not cometlogger

## 1. Load Your Data

In [ ]:
# ── Paths — edit these if your files are elsewhere ────────────────────────────
DATA_CSV    = f'{DATA_DIR}/splotopen_global.csv'
SPLITS_JSON = f'{DATA_DIR}/splotopen_global_splits.json'

plants = pd.read_csv(DATA_CSV)
plants = plants.reset_index(drop=True)   # ensure clean 0-based index

with open(SPLITS_JSON) as f:
    plants_splits = json.load(f)

print(f"Loaded {len(plants):,} rows × {len(plants.columns):,} columns")
print(f"Split keys: {list(plants_splits.keys())}")
plants.head(2)

Loaded 95,104 rows × 1,232 columns
Split keys: ['num_rows', 'meta', 'train', 'val', 'test']


,time,latitude,longitude,env_bio01,env_bio02,env_bio03,env_bio04,env_bio05,env_bio06,env_bio07,...,Erigeron uniflorus,Impatiens meruensis,Geranium sanguineum,Chamaerhodos erecta,Geranium pratense,Cassytha filiformis,Rytidosperma setaceum,Serenoa repens,Hydrocotyle vulgaris,Phillyrea angustifolia
0,0.0,62.42,-154.18,-2.608333,10.45,25.059952,1163.3374,18.5,-23.2,41.7,...,0,0,0,0,0,0,0,0,0,0
1,0.0,62.42,-154.18,-2.608333,10.45,25.059952,1163.3374,18.5,-23.2,41.7,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Identify column groups
env_cols     = [c for c in plants.columns if c.startswith('env_')]
species_cols = [c for c in plants.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

# Split env into worldclim (bio) and soilgrid
worldclim_cols = [c for c in env_cols if 'bio' in c.lower()]
soilgrid_cols  = [c for c in env_cols if 'bio' not in c.lower()]

print(f"WorldClim cols ({len(worldclim_cols)}): {worldclim_cols}")
print(f"SoilGrid cols  ({len(soilgrid_cols)}):  {soilgrid_cols}")
print(f"Species cols   ({len(species_cols)}):   {species_cols[:5]} ...")

WorldClim cols (19): ['env_bio01', 'env_bio02', 'env_bio03', 'env_bio04', 'env_bio05', 'env_bio06', 'env_bio07', 'env_bio08', 'env_bio09', 'env_bio10', 'env_bio11', 'env_bio12', 'env_bio13', 'env_bio14', 'env_bio15', 'env_bio16', 'env_bio17', 'env_bio18', 'env_bio19']
SoilGrid cols  (9):  ['env_bdod', 'env_cec', 'env_cfvo', 'env_clay', 'env_nitrogen', 'env_phh2o', 'env_sand', 'env_silt', 'env_dem']
Species cols   (1201):   ['Vaccinium myrtillus', 'Deschampsia flexuosa', 'Anthoxanthum odoratum', 'Festuca rubra', 'Achillea millefolium'] ...


## 2. Build Split Indices

In [ ]:
# Inspect what split values look like
print("First 5 train values:", plants_splits['train'][:5])
print("Type:", type(plants_splits['train'][0]))
print(f"train={len(plants_splits['train']):,}  "
      f"val={len(plants_splits['val']):,}  "
      f"test={len(plants_splits['test']):,}")
print(f"Total: {len(plants_splits['train'])+len(plants_splits['val'])+len(plants_splits['test']):,}  "
      f"vs rows: {len(plants):,}")

First 5 train values: [0, 1, 2, 3, 4]
Type: <class 'int'>
train=76,699  val=8,636  test=9,769
Total: 95,104  vs rows: 95,104


In [ ]:
# Build numpy index arrays
train_split = np.array(plants_splits['train'])
val_split   = np.array(plants_splits['val'])
test_split  = np.array(plants_splits['test'])

print(f"train_split: {train_split.shape}, max={train_split.max()}")
print(f"val_split:   {val_split.shape},   max={val_split.max()}")
print(f"test_split:  {test_split.shape},  max={test_split.max()}")
assert train_split.max() < len(plants), "Train index out of bounds!"
assert val_split.max()   < len(plants), "Val index out of bounds!"
assert test_split.max()  < len(plants), "Test index out of bounds!"
print("✓ All indices in bounds")

train_split: (76699,), max=95103
val_split:   (8636,),   max=94975
test_split:  (9769,),  max=94931
✓ All indices in bounds


## 3. Prepare Files in CISO's Expected Format

In [ ]:
# Column rename maps: your names → CISO expected names
# WorldClim: env_bio01 → bio_1, env_bio02 → bio_2, etc.
worldclim_rename = {}
for c in worldclim_cols:
    # handles env_bio01, env_bio1, env_BIO01, etc.
    num = ''.join(filter(str.isdigit, c)).lstrip('0') or '0'
    worldclim_rename[c] = f'bio_{num}'

print("WorldClim rename map:")
for k, v in worldclim_rename.items():
    print(f"  {k} → {v}")

WorldClim rename map:
  env_bio01 → bio_1
  env_bio02 → bio_2
  env_bio03 → bio_3
  env_bio04 → bio_4
  env_bio05 → bio_5
  env_bio06 → bio_6
  env_bio07 → bio_7
  env_bio08 → bio_8
  env_bio09 → bio_9
  env_bio10 → bio_10
  env_bio11 → bio_11
  env_bio12 → bio_12
  env_bio13 → bio_13
  env_bio14 → bio_14
  env_bio15 → bio_15
  env_bio16 → bio_16
  env_bio17 → bio_17
  env_bio18 → bio_18
  env_bio19 → bio_19


In [ ]:
# SoilGrid rename: adjust keys below to match your actual column names
# Run print(soilgrid_cols) if unsure
print("Your soilgrid cols:", soilgrid_cols)

# Edit this dict to match your column names on the left
soilgrid_rename = {
    # 'env_ORCDRC': 'ORCDRC',   # ← uncomment and edit as needed
    # 'env_PHIHOX': 'PHIHOX',
    # 'env_CECSOL': 'CECSOL',
    # 'env_BDTICM': 'BDTICM',
    # 'env_CLYPPT': 'CLYPPT',
    # 'env_SLTPPT': 'SLTPPT',
    # 'env_SNDPPT': 'SNDPPT',
    # 'env_BLDFIE': 'BLDFIE',
}

# Auto-build if your soilgrid cols just have an 'env_' prefix
if not soilgrid_rename:
    soilgrid_rename = {c: c.replace('env_', '').upper() for c in soilgrid_cols}
    print("Auto-built soilgrid rename map:")
    for k, v in soilgrid_rename.items():
        print(f"  {k} → {v}")

Your soilgrid cols: ['env_bdod', 'env_cec', 'env_cfvo', 'env_clay', 'env_nitrogen', 'env_phh2o', 'env_sand', 'env_silt', 'env_dem']
Auto-built soilgrid rename map:
  env_bdod → BDOD
  env_cec → CEC
  env_cfvo → CFVO
  env_clay → CLAY
  env_nitrogen → NITROGEN
  env_phh2o → PHH2O
  env_sand → SAND
  env_silt → SILT
  env_dem → DEM


In [ ]:
OUT_DIR = Path("data/sPlotOpen")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Add row ID column
plants['PlotObservationID'] = plants.index

# ── WorldClim CSV ────────────────────────────────────────────────────────────
plants[['PlotObservationID'] + worldclim_cols]\
    .rename(columns=worldclim_rename)\
    .to_csv(OUT_DIR / "worldclim_data.csv", index=False)

# ── SoilGrid CSV ─────────────────────────────────────────────────────────────
plants[['PlotObservationID'] + soilgrid_cols]\
    .rename(columns=soilgrid_rename)\
    .to_csv(OUT_DIR / "soilgrid_data.csv", index=False)

# ── Filter species: >= 100 occurrences ───────────────────────────────────────
targets_full         = plants[species_cols].to_numpy().astype(np.float32)
species_counts       = targets_full.sum(axis=0)
keep                 = species_counts >= 100
targets              = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f"Species before filtering: {len(species_cols):,}")
print(f"Species after  filtering: {len(species_cols_filtered):,}")

# ── Species list CSV ─────────────────────────────────────────────────────────
pd.DataFrame({'species': species_cols_filtered})\
    .to_csv(OUT_DIR / "species_merge_duplicates_v2.csv", index=False)

# ── Species occurrences .npy ─────────────────────────────────────────────────
np.save(OUT_DIR / "merged_species_occurrences_v2.npy", targets)

# ── Split indices .npy ───────────────────────────────────────────────────────
np.save(OUT_DIR / "train_indices.npy",      train_split)
np.save(OUT_DIR / "validation_indices.npy", val_split)
np.save(OUT_DIR / "test_indices.npy",       test_split)

print("\nFiles saved:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")

Species before filtering: 1,201
Species after  filtering: 1,201

Files saved:
  merged_species_occurrences_v2.npy
  soilgrid_data.csv
  species_merge_duplicates_v2.csv
  test_indices.npy
  train_indices.npy
  validation_indices.npy
  worldclim_data.csv


## 4. Verify Data Integrity

In [ ]:
wc  = pd.read_csv(OUT_DIR / "worldclim_data.csv")
sg  = pd.read_csv(OUT_DIR / "soilgrid_data.csv")
sp  = pd.read_csv(OUT_DIR / "species_merge_duplicates_v2.csv")
tgt = np.load(OUT_DIR / "merged_species_occurrences_v2.npy")
tr  = np.load(OUT_DIR / "train_indices.npy")
val = np.load(OUT_DIR / "validation_indices.npy")
te  = np.load(OUT_DIR / "test_indices.npy")

print(f"worldclim  : {wc.shape}   cols: {wc.columns.tolist()}")
print(f"soilgrid   : {sg.shape}   cols: {sg.columns.tolist()}")
print(f"species    : {sp.shape}   cols: {sp.columns.tolist()}")
print(f"targets    : {tgt.shape}")
print(f"train/val/test: {tr.shape} / {val.shape} / {te.shape}")

# Assertions
n_env = len(wc.columns) - 1 + len(sg.columns) - 1   # minus PlotObservationID
assert tgt.shape[1] == len(sp), \
    f"Species mismatch: targets has {tgt.shape[1]} but species list has {len(sp)}"
assert len(wc) == len(tgt), \
    f"Row mismatch: worldclim {len(wc)} vs targets {len(tgt)}"
assert tr.max() < len(tgt) and val.max() < len(tgt) and te.max() < len(tgt), \
    "Split index out of bounds!"

print(f"\n✓ All checks passed")
print(f"  input_dim  (for config) = {n_env}")
print(f"  num_classes (for config) = {tgt.shape[1]}")

worldclim  : (95104, 20)   cols: ['PlotObservationID', 'bio_1', 'bio_2', 'bio_3', 'bio_4', 'bio_5', 'bio_6', 'bio_7', 'bio_8', 'bio_9', 'bio_10', 'bio_11', 'bio_12', 'bio_13', 'bio_14', 'bio_15', 'bio_16', 'bio_17', 'bio_18', 'bio_19']
soilgrid   : (95104, 10)   cols: ['PlotObservationID', 'BDOD', 'CEC', 'CFVO', 'CLAY', 'NITROGEN', 'PHH2O', 'SAND', 'SILT', 'DEM']
species    : (1201, 1)   cols: ['species']
targets    : (95104, 1201)
train/val/test: (76699,) / (8636,) / (9769,)

✓ All checks passed
  input_dim  (for config) = 28
  num_classes (for config) = 1201


## 5. Write Config File

In [ ]:
# Build the env_columns list from the renamed columns (excluding PlotObservationID)
env_columns_for_config = (
    [worldclim_rename[c] for c in worldclim_cols] +
    [soilgrid_rename[c]  for c in soilgrid_cols]
)
print(f"env_columns ({len(env_columns_for_config)}):")
print(env_columns_for_config)

env_columns (28):
['bio_1', 'bio_2', 'bio_3', 'bio_4', 'bio_5', 'bio_6', 'bio_7', 'bio_8', 'bio_9', 'bio_10', 'bio_11', 'bio_12', 'bio_13', 'bio_14', 'bio_15', 'bio_16', 'bio_17', 'bio_18', 'bio_19', 'BDOD', 'CEC', 'CFVO', 'CLAY', 'NITROGEN', 'PHH2O', 'SAND', 'SILT', 'DEM']


In [ ]:
CONFIG_PATH = Path("configs/config_ciso_mydata.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Edit these as needed ──────────────────────────────────────────────────────
EXPERIMENT_NAME = "splot_ciso_mydata"
CHECKPOINT_DIR  = CKPT_DIR
MAX_EPOCHS      = 50
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

config_dict = {
    'mode': 'train',
    'dataset_name': 'sPlot',
    'model': {
        'name': 'CISOModel',
        'input_dim': n_env,
        'hidden_dim': 256,
        'num_classes': int(tgt.shape[1]),
        'backbone': 'SimpleMLPBackbone',
    },
    'training': {
        'seed': 1339,
        'learning_rate': LEARNING_RATE,
        'max_epochs': MAX_EPOCHS,
        'accelerator': 'gpu',
        'devices': 1,
    },
    'logger': {
        'project_name': 'sPlotOpen',
        'experiment_name': EXPERIMENT_NAME,
        'experiment_key': '',
        'checkpoint_path': CHECKPOINT_DIR,
        'checkpoint_name': '',
        'save_preds_path': '',
    },
    'data': {
        'dataloader_to_use': 'sPlotMaskedDataset',
        'base': 'data/sPlotOpen',
        'train': 'train_indices.npy',
        'validation': 'validation_indices.npy',
        'test': 'test_indices.npy',
        'targets': 'merged_species_occurrences_v2.npy',
        'worldclim_data_path': 'worldclim_data.csv',
        'soilgrid_data_path': 'soilgrid_data.csv',
        'species_list': 'species_merge_duplicates_v2.csv',
        'species_occurrences_threshold': 100,
        'batch_size': BATCH_SIZE,
        'env_columns': env_columns_for_config,
        'partial_labels': {
            'use': True,
            'quantized_mask_bins': 1,      # binary presence/absence
            'train_known_ratio': 0.75,     # max 75% species known during training
            'eval_known_ratio': 0,         # fully unconditioned at eval
            'predict_family_of_species': -1,  # -1 = predict all species
        },
    },
}

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {CONFIG_PATH}")
print()
print(open(CONFIG_PATH).read())

Config written to: configs/config_ciso_mydata.yaml

mode: train
dataset_name: sPlot
model:
  name: CISOModel
  input_dim: 28
  hidden_dim: 256
  num_classes: 1201
  backbone: SimpleMLPBackbone
training:
  seed: 1339
  learning_rate: 0.001
  max_epochs: 50
  accelerator: gpu
  devices: 1
logger:
  project_name: sPlotOpen
  experiment_name: splot_ciso_mydata
  experiment_key: ''
  checkpoint_path: /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339
  checkpoint_name: ''
  save_preds_path: ''
data:
  dataloader_to_use: sPlotMaskedDataset
  base: data/sPlotOpen
  train: train_indices.npy
  validation: validation_indices.npy
  test: test_indices.npy
  targets: merged_species_occurrences_v2.npy
  worldclim_data_path: worldclim_data.csv
  soilgrid_data_path: soilgrid_data.csv
  species_list: species_merge_duplicates_v2.csv
  species_occurrences_threshold: 100
  batch_size: 64
  env_columns:
  - bio_1
  - bio_2
  - bio_3
  - bio_4
  - bio_5
  - bio_6
  - bio_7
  - bio_8
  - bio_9
  - 

## 6. Train

In [ ]:
import os
os.environ['COMET_MODE'] = 'OFFLINE'
os.environ.setdefault('COMET_OFFLINE_DIRECTORY', 'comet_logs')
os.makedirs('comet_logs', exist_ok=True)

In [ ]:
# Verify config looks right before launching
!python main.py --help

usage: main.py [-h] --config CONFIG [--run_id RUN_ID]
               [--results_file_name RESULTS_FILE_NAME]

PyTorch Lightning Tabular Data MLP Training

options:
  -h, --help            show this help message and exit
  --config CONFIG
  --run_id RUN_ID
  --results_file_name RESULTS_FILE_NAME


In [ ]:
!python main.py --config configs/config_ciso_mydata.yaml

/content/CISO-SDM/configs/config_ciso_mydata.yaml
Mode: train
Model Config: name='CISOModel' input_dim=28 hidden_dim=256 num_classes=1201 backbone='SimpleMLPBackbone'
Data Path Config: dataloader_to_use='sPlotMaskedDataset' base='data/sPlotOpen' maxent_transform=False train='train_indices.npy' validation='validation_indices.npy' test='test_indices.npy' targets='merged_species_occurrences_v2.npy' worldclim_data_path='worldclim_data.csv' soilgrid_data_path='soilgrid_data.csv' species_occurrences_threshold=100 batch_size=64 species_list='species_merge_duplicates_v2.csv' env_columns=['bio_1', 'bio_2', 'bio_3', 'bio_4', 'bio_5', 'bio_6', 'bio_7', 'bio_8', 'bio_9', 'bio_10', 'bio_11', 'bio_12', 'bio_13', 'bio_14', 'bio_15', 'bio_16', 'bio_17', 'bio_18', 'bio_19', 'BDOD', 'CEC', 'CFVO', 'CLAY', 'NITROGEN', 'PHH2O', 'SAND', 'SILT', 'DEM'] partial_labels=PartialLabels(use=True, predict_family_of_species=-1, train_known_ratio=0.75, eval_known_ratio=0.0, quantized_mask_bins=1)
Training Config: se

In [ ]:
# Check what checkpoint was saved
ckpt_dir = Path(CHECKPOINT_DIR)
if ckpt_dir.exists():
    ckpts = list(ckpt_dir.glob("**/*.ckpt")) + list(ckpt_dir.glob("**/*.pt"))
    print("Saved checkpoints:")
    for c in ckpts:
        print(f"  {c}")
else:
    print(f"Checkpoint dir not found: {ckpt_dir}")

Saved checkpoints:
  /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339/splot_ciso_mydata/1339/epoch=33-step=40766.ckpt
  /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339/splot_ciso_mydata/1339/last.ckpt
  /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339/splot_ciso_mydata/1339/last-v1.ckpt
  /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339/splot_ciso_mydata/1339/epoch=43-step=52756.ckpt


## 7. Benchmark at Multiple Masking Levels

Evaluate CISO at `eval_known_ratio` ∈ {0.0, 0.25, 0.5, 0.75, 1.0}, corresponding to masking fractions p ∈ {1.0, 0.75, 0.5, 0.25, 0.0} — matching the four masking levels reported in STEM-LM.

In [ ]:
# Random-mask sweep matching STEM-LM's --val_p_list. p=0 (eval_known_ratio=1.0)
# is omitted: all species revealed = trivial copy-through, not a real prediction.
EVAL_RATIOS = [0.0, 0.25, 0.5, 0.75]

# Auto-detect checkpoint
CHECKPOINT_NAME = ""
ckpts = list(Path(CKPT_DIR).glob("**/*.ckpt"))
best = [p for p in ckpts if 'last' not in p.name]
if best:
    CHECKPOINT_NAME = str(sorted(best)[-1])
elif ckpts:
    CHECKPOINT_NAME = str(sorted(ckpts)[-1])

if CHECKPOINT_NAME:
    print(f"Auto-detected checkpoint: {CHECKPOINT_NAME}")
else:
    print("No checkpoint found — run training first")


Auto-detected checkpoint: /content/drive/MyDrive/CISO/model_checkpoints_50_epochs_1339/splot_ciso_mydata/1339/epoch=43-step=52756.ckpt


In [ ]:
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

for ratio in EVAL_RATIOS:
    print(f"\n{'='*60}")
    print(f"Evaluating eval_known_ratio={ratio}  (mask p={1-ratio:.2f})")
    print('='*60, flush=True)

    ratio_str = str(ratio).replace('.', 'p')

    # Create a directory for this ratio's prediction shards
    preds_dir = f'{RESULTS_DIR}/preds_ratio_{ratio_str}'
    Path(preds_dir).mkdir(parents=True, exist_ok=True)   # ← fix

    test_config = config_dict.copy()
    test_config['mode'] = 'test'
    test_config['logger'] = config_dict['logger'].copy()
    test_config['logger']['checkpoint_name'] = CHECKPOINT_NAME
    test_config['logger']['save_preds_path'] = preds_dir  # ← directory, not .npy file
    test_config['data'] = config_dict['data'].copy()
    test_config['data']['partial_labels'] = config_dict['data']['partial_labels'].copy()
    test_config['data']['partial_labels']['eval_known_ratio'] = ratio

    cfg_path = f"configs/config_ciso_test_{ratio_str}.yaml"
    res_csv  = f"{RESULTS_DIR}/results_ratio_{ratio_str}.csv"

    with open(cfg_path, 'w') as f:
        yaml.dump(test_config, f, default_flow_style=False, sort_keys=False)

    !python main.py --config {cfg_path} --results_file_name {res_csv}

print("\nDone — all masking levels evaluated.")


Evaluating eval_known_ratio=0.0  (mask p=1.00)
/content/CISO-SDM/configs/config_ciso_test_0p0.yaml
Mode: test
Model Config: name='CISOModel' input_dim=28 hidden_dim=256 num_classes=1201 backbone='SimpleMLPBackbone'
Data Path Config: dataloader_to_use='sPlotMaskedDataset' base='data/sPlotOpen' maxent_transform=False train='train_indices.npy' validation='validation_indices.npy' test='test_indices.npy' targets='merged_species_occurrences_v2.npy' worldclim_data_path='worldclim_data.csv' soilgrid_data_path='soilgrid_data.csv' species_occurrences_threshold=100 batch_size=64 species_list='species_merge_duplicates_v2.csv' env_columns=['bio_1', 'bio_2', 'bio_3', 'bio_4', 'bio_5', 'bio_6', 'bio_7', 'bio_8', 'bio_9', 'bio_10', 'bio_11', 'bio_12', 'bio_13', 'bio_14', 'bio_15', 'bio_16', 'bio_17', 'bio_18', 'bio_19', 'BDOD', 'CEC', 'CFVO', 'CLAY', 'NITROGEN', 'PHH2O', 'SAND', 'SILT', 'DEM'] partial_labels=PartialLabels(use=True, predict_family_of_species=-1, train_known_ratio=0.75, eval_known_rati

## 8. Summarise Results

In [ ]:
import os, yaml, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from src.config import Config
from src.dataloaders.splot_dataloader import sPlotDataModule
from src.trainers.splot_trainer import sPlotTrainer

# ── Vendored from STEMLM_metric.py (verbatim) ───────────────────────────
def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')


def _per_species_metrics(probs, targets, masks):
    """Per-species metrics, FILTERED to rows where each species was masked.
    probs, targets, masks: (N, S) float / int / int. mask == -1 => masked target."""
    S = probs.shape[1]
    out = {k: {} for k in ['auc_roc','auc_pr','cbi','brier','ece']}
    n_kept_per_sp = []
    for s in range(S):
        keep = masks[:, s] == -1
        n_kept_per_sp.append(int(keep.sum()))
        if keep.sum() == 0:
            continue
        y = targets[keep, s].astype(np.int64)
        p = probs[keep, s].astype(np.float64)
        if y.sum() == 0 or y.sum() == y.size:
            continue
        out['auc_roc'][s] = _safe_auc_roc(y, p)
        out['auc_pr'][s]  = _safe_auc_pr(y, p)
        out['cbi'][s]     = _safe_cbi(y, p)
        out['brier'][s]   = _safe_brier(y, p)
        out['ece'][s]     = _safe_ece(y, p)
    return out, n_kept_per_sp

def _summarize(per_sp):
    def cl(d): return [v for v in d.values() if np.isfinite(v)]
    aucs = cl(per_sp['auc_roc']); prs = cl(per_sp['auc_pr'])
    cbis = cl(per_sp['cbi']);     bri = cl(per_sp['brier']);  ece = cl(per_sp['ece'])
    q = lambda x, p: float(np.quantile(x, p)) if x else float('nan')
    return {
        'mean_auc_roc': float(np.mean(aucs)) if aucs else float('nan'),
        'auc_roc_q25': q(aucs, .25), 'auc_roc_q50': q(aucs, .50), 'auc_roc_q75': q(aucs, .75),
        'mean_auc_pr': float(np.mean(prs)) if prs else float('nan'),
        'mean_cbi':    float(np.mean(cbis)) if cbis else float('nan'),
        'mean_brier':  float(np.mean(bri)) if bri else float('nan'),
        'mean_ece':    float(np.mean(ece)) if ece else float('nan'),
        'n_species':   len(aucs),
    }


def _get_seed(run_id, seed): return (run_id * (seed + (run_id - 1))) % (2**31 - 1)

def _run_inference_with_masks(test_cfg_path):
    """Load CISO checkpoint + run on test loader; return (probs, targets, masks) numpy.
    Captures the per-row mask tensor produced by sPlotMaskedDataset so that
    metrics can be filtered to only-masked species (matching STEM-LM).
    GPU-enabled: model and batches moved to CUDA when available."""
    cfg  = Config(**yaml.safe_load(open(test_cfg_path)))
    seed = _get_seed(1, cfg.training.seed)
    ckpt = os.path.join(cfg.logger.checkpoint_path, cfg.logger.experiment_name,
                        str(seed), cfg.logger.checkpoint_name)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    dm   = sPlotDataModule(cfg.data); dm.setup()
    task = sPlotTrainer(cfg).to(device)
    task.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'])
    task.eval()

    probs_all, tgt_all, mask_all = [], [], []
    with torch.no_grad():
        for batch in dm.test_dataloader(num_workers=0, persistent_workers=False):
            batch_dev = {k: (v.to(device) if torch.is_tensor(v) else v)
                         for k, v in batch.items()}
            logits = task(batch_dev)
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            tgt_all.append(batch_dev['targets'].cpu().numpy())
            mask_all.append(batch_dev['mask'].cpu().numpy())  # -1 = masked, 0/1 = known
    return (np.concatenate(probs_all, 0),
            np.concatenate(tgt_all, 0).astype(np.int64),
            np.concatenate(mask_all, 0).astype(np.int64))


# ── Sweep: full STEM-LM metric set per eval_known_ratio ─────────────────
rows = []
for ratio in EVAL_RATIOS:
    rs = str(ratio).replace('.', 'p')
    cfg_path = f"configs/config_ciso_test_{rs}.yaml"
    if not Path(cfg_path).exists():
        print(f"missing {cfg_path} — skipping"); continue
    print(f"\n→ ratio={ratio} (mask p={1-ratio:.2f})", flush=True)
    probs, tgt, mask = _run_inference_with_masks(cfg_path)
    per_sp, n_kept = _per_species_metrics(probs, tgt, mask)
    summ = _summarize(per_sp)
    summ.update({'eval_known_ratio': ratio, 'masking_p': round(1-ratio, 2),
                 'avg_masked_rows_per_sp': float(np.mean(n_kept))})
    rows.append(summ)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in summ.items()})

summary = pd.DataFrame(rows).set_index('masking_p').sort_index()
cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species','avg_masked_rows_per_sp']
summary = summary[['eval_known_ratio'] + cols]
print("\n── CISO benchmark (STEM-LM-faithful, masked-only filter) ─────────")
print(summary.to_string())
out_csv = f"{RESULTS_DIR}/ciso_benchmark_summary.csv"
summary.to_csv(out_csv)
print(f"\nSaved to {out_csv}")


→ ratio=0.0 (mask p=1.00)
Number of classes: 1201


/tmp/ipykernel_1921/2890247813.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'])


{'mean_auc_roc': 0.9432, 'auc_roc_q25': 0.9209, 'auc_roc_q50': 0.9528, 'auc_roc_q75': 0.979, 'mean_auc_pr': 0.1873, 'mean_cbi': 0.5164, 'mean_brier': 0.0089, 'mean_ece': 0.0057, 'n_species': 1050, 'eval_known_ratio': 0.0, 'masking_p': 1.0, 'avg_masked_rows_per_sp': 9769.0}

→ ratio=0.25 (mask p=0.75)
Number of classes: 1201


/tmp/ipykernel_1921/2890247813.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'])


{'mean_auc_roc': 0.9626, 'auc_roc_q25': 0.9517, 'auc_roc_q50': 0.9703, 'auc_roc_q75': 0.9844, 'mean_auc_pr': 0.2794, 'mean_cbi': 0.6828, 'mean_brier': 0.008, 'mean_ece': 0.0045, 'n_species': 1050, 'eval_known_ratio': 0.25, 'masking_p': 0.75, 'avg_masked_rows_per_sp': 8543.5437}

→ ratio=0.5 (mask p=0.50)
Number of classes: 1201


/tmp/ipykernel_1921/2890247813.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'])


{'mean_auc_roc': 0.9672, 'auc_roc_q25': 0.9584, 'auc_roc_q50': 0.9755, 'auc_roc_q75': 0.9863, 'mean_auc_pr': 0.3105, 'mean_cbi': 0.7052, 'mean_brier': 0.0078, 'mean_ece': 0.0045, 'n_species': 1039, 'eval_known_ratio': 0.5, 'masking_p': 0.5, 'avg_masked_rows_per_sp': 7332.164}

→ ratio=0.75 (mask p=0.25)
Number of classes: 1201


/tmp/ipykernel_1921/2890247813.py:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'])


{'mean_auc_roc': 0.9688, 'auc_roc_q25': 0.9613, 'auc_roc_q50': 0.9768, 'auc_roc_q75': 0.9881, 'mean_auc_pr': 0.3302, 'mean_cbi': 0.6949, 'mean_brier': 0.0076, 'mean_ece': 0.0045, 'n_species': 1038, 'eval_known_ratio': 0.75, 'masking_p': 0.25, 'avg_masked_rows_per_sp': 6116.3913}

── CISO benchmark (STEM-LM-faithful, masked-only filter) ─────────
           eval_known_ratio  mean_auc_roc  auc_roc_q25  auc_roc_q50  auc_roc_q75  mean_auc_pr  mean_cbi  mean_brier  mean_ece  n_species  avg_masked_rows_per_sp
masking_p                                                                                                                                                       
0.25                   0.75      0.968826     0.961328     0.976774     0.988144     0.330200  0.694893    0.007615  0.004480       1038             6116.391341
0.50                   0.50      0.967211     0.958445     0.975488     0.986326     0.310522  0.705209    0.007765  0.004460       1039             7332.164030
0.75    